# S4 prepare features


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("bmi", StandardScaler(), ["BMI"])
    ],
    remainder="passthrough"
)

# S5 Train logistic regression
todo legg til forklaring/kommentering

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression


trainSet = pd.read_csv("generated/train_set.csv")
testSet = pd.read_csv("generated/test_set.csv")


X_train = trainSet.drop(columns=["Diabetes_binary"])
y_train = trainSet["Diabetes_binary"]

X_test = testSet.drop(columns=["Diabetes_binary"])
y_test = testSet["Diabetes_binary"]

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)


model.fit(X_train, y_train)


predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

# S6 Evaluating the Model



After using the model, we must asses its performance.

In [ ]:
import matplotlib.pyplot as plt
from sklearn import metrics

#TODO: Replace with model outputs
yTrue = y_test #True answers, as per the train dataset
yPredicted = predictions #Answers predicted by the model

## Confusion Matrix

Generating a confusion matrix, we can find out which values the model got correct and which wrong.

Correct answers are a *true positive (TP)* or *true negative (TN)*.

Incorrect answers are:
- **False Positive (FP)**: A patient was flagged as having diabetes, despite them not really having it. While this causes loss of time and some resources, it is not strictly dangerous as more research will most likely be performed to debunk the case.
- **False Negative (FN)**: A patient who does have diabetes was flagged as not having it. Contrary to an FP, this is dangerous, and further reasearch is quite likely to not happen at all.

The confusion matrix is explored later to get data on the percentages of answers the model got right.

In [ ]:
cm = metrics.confusion_matrix(yTrue, yPredicted)

TN = cm[0, 0]
FN = cm[1, 0]
TP = cm[1, 1]
FP = cm[0, 1]

disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

## Accuracy

How many answers did the model get right?

If the model performs well enough, this value will usually be quite high.
Note that accuracy is, in fact, not an accurate metric. The reason simply is that if the TP case is rare and the model usually predicts false, then it will have a high score as most cases were indeed false.
Put differently: If the model just always says false and 1 out of 10 cases are true, the accuracy will end up as 90%.

In [ ]:
accuracy = (TP + TN) / (TP + TN + FP + FN)

print(accuracy)

## Precision


How many did the model flag who actually have diabetes?

Precision is about checking how often the model gives a false positive. That is, how often does somebody without diabetes get flagged as having it.
Simply put: How reliable is the model when it predicts true.

In [ ]:
precision = TP / (TP + FP)

print(precision)

## Recall

How many who have diabetes were correctly identified by the model?

Recall is about checking how many of the cases the model has correctly caught. If this value is at 100%, it means that there were no false negatives, and thus the model got every real case right.

`1 - recall` would on the other hand tell us how many true cases the model had missed.

In [ ]:
recall = TP / (TP + FN)

print(recall)

## F1 Score

F1 score summarizes *precision* and *recall* into one number as a harmonic mean.
A value of *100%* means perfect *precision* and *recall*. 

In [ ]:
f1_score = 2 * (precision * recall) / (precision + recall)

print(f1_score)

# S7 Probebility threshold

In [ ]:
#TODO: Insert the model. It is a complete pipeline object
probabilities = model.predict_proba(X_test)[:, 1] #Prediction values for class 1

def threshold(probs, t):
    #Return boolean mask of where diabetes cases are likely enough
    #Size is the same
    return probs >= t 

cases_thresh_05 = threshold(probabilities, 0.5)
cases_thresh_04 = threshold(probabilities, 0.4)

def plot_cm(y_true, y_predicted, title):
    cm = metrics.confusion_matrix(y_true, y_predicted, labels=[0, 1])

    metrics.ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No diabetes", "Diabetes"]
    ).plot()

    plt.title(title)
    plt.show()

    return cm


In [ ]:
cm_05 = plot_cm(y_test, cases_thresh_05, "Threshold 0.50")
cm_04 = plot_cm(y_test, cases_thresh_04, "Threshold 0.40")

In [ ]:
comparison = pd.DataFrame({
    "Threshold": [0.50, 0.40],
    "Accuracy": [
        metrics.accuracy_score(y_test, cases_thresh_05),
        metrics.accuracy_score(y_test, cases_thresh_04)
    ],
    "Precision": [
        metrics.precision_score(y_test, cases_thresh_05),
        metrics.precision_score(y_test, cases_thresh_04)
    ],
    "Recall": [
        metrics.recall_score(y_test, cases_thresh_05),
        metrics.recall_score(y_test, cases_thresh_04)
    ],
    "F1": [
        metrics.f1_score(y_test, cases_thresh_05),
        metrics.f1_score(y_test, cases_thresh_04)
    ],
    "False positives": [cm_05[0, 1], cm_04[0, 1]],
    "False negatives": [cm_05[1, 0], cm_04[1, 0]]
})

display(comparison.round(4))